# Scrape CWE

### libraries

In [1]:
%pip install beautifulsoup4 networkx requests tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
from bs4 import BeautifulSoup, Tag
import requests
import json
from tqdm import tqdm

In [3]:
# This is the url that points to the top categories and children
response = requests.get("https://cwe.mitre.org/data/definitions/699.html")
data = response.content
html = BeautifulSoup(data)

In [4]:
def extract_general_info(group: Tag):
    title_html = group.select_one("span.Primary>a")
    if not title_html:
        raise ValueError(f"Tried grabbing title but it returned none on {group}")
    
    title = title_html.text.split(" - ")[0].strip()
    id = int(title_html.text.split(" - ")[-1].strip().removeprefix("(").removesuffix(")"))
    title_url = base_url + str(title_html.get("href", ""))
    
    return title, id, title_url

In [5]:
base_url = "https://cwe.mitre.org"
categories_html = html.select(
    "#oc_699_Relationships > div > div > div > div:nth-child(5)>div.group"
)
print(str(len(categories_html)) + " Categories")

categories = {}

for category_html in categories_html:
    # Retrieving the category information
    category_title, category_id, category_url = extract_general_info(category_html)

    # Retrieving the children information
    children_html = category_html.select(".group")
    children = {}
    for child_html in children_html:
        child_title, child_id, child_url = extract_general_info(child_html)
        children[child_id] = {"id": child_id, "title": child_title, "url": child_url, "type": "child"}

    categories[category_id] = {
        "id": category_id,
        "title": category_title,
        "url": category_url,
        "children": children,
        "type": "category"
    }

def print_cat():
    print(json.dumps(list(categories.values())[0], indent=4))
    
print_cat()

40 Categories
{
    "id": 1228,
    "title": "API / Function Errors",
    "url": "https://cwe.mitre.org/data/definitions/1228.html",
    "children": {
        "242": {
            "id": 242,
            "title": "Use of Inherently Dangerous Function",
            "url": "https://cwe.mitre.org/data/definitions/242.html",
            "type": "child"
        },
        "474": {
            "id": 474,
            "title": "Use of Function with Inconsistent Implementations",
            "url": "https://cwe.mitre.org/data/definitions/474.html",
            "type": "child"
        },
        "475": {
            "id": 475,
            "title": "Undefined Behavior for Input to API",
            "url": "https://cwe.mitre.org/data/definitions/475.html",
            "type": "child"
        },
        "477": {
            "id": 477,
            "title": "Use of Obsolete Function",
            "url": "https://cwe.mitre.org/data/definitions/477.html",
            "type": "child"
        },
        "

### Scraping the summary for each category

In [6]:
def scrape_category_summary(url: str) -> str:
    response = requests.get(url)
    html = BeautifulSoup(response.content)
    summary = html.select_one("#Summary .indent")
    if not summary:
        raise ValueError(f"Something went wrong cuz I can't find the summary text {url}")
    
    
    return summary.text.strip()

scrape_category_summary("https://cwe.mitre.org/data/definitions/1228.html")
    

'Weaknesses in this category are related to the use of built-in functions or external APIs.'

In [7]:
# Scraping the summary for every category and updating the dictionary

for id, category in tqdm(categories.items()):
    category["summary"] = scrape_category_summary(category["url"])
    
print_cat()

100%|██████████| 40/40 [00:10<00:00,  3.97it/s]

{
    "id": 1228,
    "title": "API / Function Errors",
    "url": "https://cwe.mitre.org/data/definitions/1228.html",
    "children": {
        "242": {
            "id": 242,
            "title": "Use of Inherently Dangerous Function",
            "url": "https://cwe.mitre.org/data/definitions/242.html",
            "type": "child"
        },
        "474": {
            "id": 474,
            "title": "Use of Function with Inconsistent Implementations",
            "url": "https://cwe.mitre.org/data/definitions/474.html",
            "type": "child"
        },
        "475": {
            "id": 475,
            "title": "Undefined Behavior for Input to API",
            "url": "https://cwe.mitre.org/data/definitions/475.html",
            "type": "child"
        },
        "477": {
            "id": 477,
            "title": "Use of Obsolete Function",
            "url": "https://cwe.mitre.org/data/definitions/477.html",
            "type": "child"
        },
        "676": {
      

### Scraping summary and information about children

In [8]:
def scrape_child(url: str):
    response = requests.get(url)
    html = BeautifulSoup(response.content)
    
    # Description
    description_html = html.select_one("#Description .indent")
    if not description_html:
        raise ValueError(f"could not find description for {url}")

    description = description_html.text.strip()
    
    # Extended description
    extended_description_html = html.select_one("#Extended_Description .indent")
    if not extended_description_html:
        extended_description = None
    else:
        extended_description = extended_description_html.text.strip()
    
    return description, extended_description

scrape_child("https://cwe.mitre.org/data/definitions/242.html")

('The product calls a function that can never be guaranteed to work safely.',
 'Certain functions behave in dangerous ways regardless of how they are used. Functions in this category were often implemented without taking security concerns into account. The gets() function is unsafe because it does not perform bounds checking on the size of its input. An attacker can easily send arbitrarily-sized input to gets() and overflow the destination buffer. Similarly, the >> operator is unsafe to use when reading into a statically-allocated character array because it does not perform bounds checking on the size of its input. An attacker can easily send arbitrarily-sized input to the >> operator and overflow the destination buffer.')

In [9]:
for category in tqdm(categories.values(), position=0, desc="Categories"):
    for child in category["children"].values():
        description, extended_description = scrape_child(child["url"])
        child["description"] = description
        child["extended_description"] = extended_description

Categories: 100%|██████████| 40/40 [02:39<00:00,  3.98s/it]


In [10]:
with open("./scraped_data/CWE_SOFTWARE.json", "w", encoding="utf-8") as f:
    json.dump(categories, f, indent=2)